# Import Statements

In [ ]:
import custom_cmap
import os
import sys
from PIL import Image
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
from moria import reduce

from pathlib import Path
from astropy.visualization import PercentileInterval

from astropy.io import fits
from astropy.visualization import LogStretch, ImageNormalize
import plotly.express as px
import numpy as np
from astropy.visualization import ImageNormalize, AsinhStretch, SqrtStretch, LogStretch, PowerStretch

plt.rcParams.update({'font.size':25})
plt.rc('text', usetex=True)
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2


# Understanding the main directory. 

The data directory should be organized as follows. You can look at the sample data folder to understand the directory strucutre you need.

<li>00.DATA</li>
<li>01.XYM</li>
<li>02.CMD</li>
<li>03.LOC_TRANS</li>
<li>04.PSF_EXTRACT</li>
<li>05.COORD_TRANS(OPTIONAL)</li>
<li>06.FIT</li>
<li>07.CALIBRATION</li>

All the necessary scripts are included in the sample data directory. 

Using the matchup files generated in "output_stacks.ipynb", a CMD file can be generated. The CMD is used to identify several important stars with colors and magnitudes similar to the target that will be used to construct a PSF model with similar CTE distortions to the target. The matchup files also select brighter stars near the targer that can be used to refine the coordinate transformation and that can be used to calibrate the HST photometry to the OGLE-III catalog. 

After you finish running this notebook, you will be able to use the "_flc" exposures to generate an output image stack in the F814W and F606W HST filters.

NOTE: chmod +x program.src is a useful command to use whenever permissions are denied for a script. 

In [ ]:
directory = os.getcwd()

# Step 0 

We assume you ran the output_stacks.ipynb notebook correctly

# Step 1

Run the python script to get the CMD. This step will also move the target star to the top in the MATCHUP files for the F814W and F606W folders generated in 01.XYM/F814W and 01.XYM/F606W.

We assume that IF you have a target star, you can use the 'outputq_F814W.fits' in 01.XYM/F814W to locate the pixel co-ordinates of your target. The pixel co-ordinates of your target identified in 'outputq_F814W.fits' should have a match (to the nearest decimal) in the MATCHUP files. The MATCHUP files will also give you the I magnitude needed to plot the CMD diagram for your target. Similar steps will have to be taken to find the V magnitude of your target by navigating 01.XYM/F606W. 


The rest of the arguments you give in the function "cmd_diagram" below are abritrary and influence your plotting choice for the CMD. 

### reduce.cmd_diagram(directory)

In [ ]:
reduce.cmd_diagram(directory)

If you wish to continue with the pipeline beyond creating this CMD diagram, it is very important that your file 'NEARBY_SIM_STARS' have between 20-40 stars. This file can be found in your 02.CMD folder. If there are more than 40 stars, you should run "reduce.cmd_diagram(directory)" with tighter constraints.


The CMD diagrams are created in the 02.CMD folder. In this step we also created thre important files: 

1. NEARBY_SIM_STARS.XYIVB_targ - This contains the stars similar to the target to be used for the PSF model construction.
2. NEARBY_REF_STARS.XYIVB_targ - This contains stars close to the target to be used to refine the coordinate transformations.
3. NOTFAR_CAL_STARS.XYIVB_targ - This contains the stars that will be calibrated to the OGLE-III catalog. Note that there aren’t very many stars that won’t be blended in the OGLE-III data, so the calibration stars should be selected over a wider range of coordinates than the candidate PSF stars.


Note that the code used to identify PSF model outliers currently only allows up to 40 PSF stars, so the NEARBY_SIM_STARS.XYIVB_targ file should be kept to a maximum of 40 stars (42 lines in the file). Let us open NEARBY_SIM_STARS.XYIVB_targ to ensure that this is the case.

In [ ]:
filename_sim_stars = Path(directory).resolve()/f"02.CMD/NEARBY_SIM_STARS.XYIVB_targ"
cols = ["xu", "yu", "miu", "mvu", "psf"]
df = pd.read_csv(filename_sim_stars, sep=r"\s+", comment="#", header=None, names=cols)

In [ ]:
df

Let us also open the file "show_cmd_targ" to inspect the CMD diagram generated around our target, along with panels showing the reference stars selected to create the "NEARBY_REF_STARS.XYIVB_targ" file

In [ ]:
import base64
from IPython.display import IFrame
pdf_path = Path(directory).resolve()/f"02.CMD/show_cmd_targ.pdf" 
with open(pdf_path, "rb") as pdf_file:
        encoded_pdf = base64.b64encode(pdf_file.read()).decode("utf-8")
IFrame(f"data:application/pdf;base64,{encoded_pdf}", width=800, height=800)


Lastly, let us open the file "show_cmd_cal" to inspect the CMD diagram generated around our target, along with panels showing the reference stars selected to create the "NEARBY_CAL_STARS.XYIVB_targ" file

In [ ]:
import base64
from IPython.display import IFrame
pdf_path = Path(directory).resolve()/f"02.CMD/show_cmd_cal.pdf" 
with open(pdf_path, "rb") as pdf_file:
        encoded_pdf = base64.b64encode(pdf_file.read()).decode("utf-8")
IFrame(f"data:application/pdf;base64,{encoded_pdf}", width=450, height=450)
